# Playground — central orchestrator

Only calls functions from `src/tools/`, `src/modelling/`, `agent/`. Heavy logic lives there.

Linear data flow:
1. **Load** raw data
2. **Clean** (split sales / returns)
3. **Features** (weekly aggregation, return rate, SKU universe)
4. **Embedding** (Gemini Embedding 2 on SKU descriptions)
5. Model selection on validation + held-out test
6. Final 12-week forecast + revenue translation
7. Plots
8. Agent-ready tables + prompt demo

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
sys.path.insert(0, str(ROOT))

# Load secrets from .env (GEMINI_API_KEY, ...). Copy .env.example to .env first.
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import pandas as pd
import matplotlib.pyplot as plt

from src.tools import (
    load_raw_data, split_sales_returns,
    aggregate_weekly_sku, median_price_per_sku,
    eligible_skus_by_revenue, return_rate_features,
    embed_sku_descriptions,
)
from src.tools import plots
from src.modelling import default_registry, select_best_model, forecast_final_horizon, attach_revenue
from agent import build_agent_tables, get_forecast_for_prompt

## 1. Load — read both Excel sheets

In [ ]:
DATA_PATH = ROOT / "Datasets" / "online_retail_II.xlsx"
df = load_raw_data(DATA_PATH)
print(f"raw rows: {len(df):,}  |  sheets: {df['SourceSheet'].unique().tolist()}")
df.head(3)

## 2. Clean — split sales (target) from returns (feature only)

In [ ]:
sales, returns = split_sales_returns(df)
print(f"sales: {len(sales):,}  |  returns: {len(returns):,}")
print(f"unique SKUs in sales: {sales['StockCode'].nunique():,}")

## 3. Features — weekly aggregation, return rate, SKU universe

In [ ]:
weekly_sku = aggregate_weekly_sku(sales)
sku_price = median_price_per_sku(sales)
rr = return_rate_features(weekly_sku, returns, windows=(4, 13))

TOP_N = 30
skus = eligible_skus_by_revenue(weekly_sku, top_n=TOP_N, min_active_weeks=60)
print(f"weekly rows: {len(weekly_sku):,}  |  SKUs to evaluate: {len(skus)}")
rr.head()

## 4. Embedding — Gemini Embedding 2 on canonical SKU descriptions

Requires `GEMINI_API_KEY` in the environment. First run hits the API and caches to parquet; subsequent runs read from cache.

```bash
export GEMINI_API_KEY="your-key-from-aistudio.google.com"
```

In [ ]:
EMB_PATH = ROOT / "data" / "processed" / "sku_desc_emb.parquet"

emb = embed_sku_descriptions(
    sales,
    cache_path=EMB_PATH,
    dim=768,
    task_type="CLUSTERING",
)
print(f"embedded SKUs: {len(emb):,}  |  dim: {len(emb['embedding'].iloc[0])}")
emb.head(3)

## 5. Model selection (validation block-MAPE) + held-out test

Each model is `Callable[[pd.Series, int], np.ndarray]` — `(train_series, horizon)` → forecast vector.
Add new ones by extending `default_registry()`.

In [ ]:
registry = default_registry()
print("Models:", list(registry))

results = select_best_model(
    weekly_sku=weekly_sku,
    skus=skus,
    model_registry=registry,
    block_size=4, val_size=12, test_size=12,
)
choices_df = results["choices"]
test_block = results["test_block"]
print("Overall test block-MAPE:", round(test_block['Block_APE'].mean(), 2))
choices_df.head()

## 6. Final 12-week forecast + revenue translation

In [ ]:
forecast_df = forecast_final_horizon(weekly_sku, choices_df, registry, horizon=12)
fc_with_rev = attach_revenue(forecast_df, sku_price)
fc_with_rev.head()

## 7. Plots

In [ ]:
plots.plot_block_ape_boxplot(test_block); plt.show()
plots.plot_chosen_model_counts(choices_df); plt.show()

## 8. Agent-ready interface

`build_agent_tables` materializes the two lookups the n8n agent reads from.
`get_forecast_for_prompt` is the function the agent's tool calls under the hood.

In [ ]:
agent_horizon, agent_summary = build_agent_tables(forecast_df, sku_price, choices_df)

for prompt in [
    "What is the latest 12-week forecast for product 22197?",
    "Forecast SKU 85123A for the next 4 weeks",
    "give me the forecast for ABCDEF",
]:
    print(get_forecast_for_prompt(prompt, agent_summary, agent_horizon))
    print("\n" + "=" * 60 + "\n")